# Week 4 - Publish the fine-tuned model on Hugging Face

Uploads the 5 GGUF levels, the LoRA adapter and a model card to a **private** Hugging Face repo.
Every file is checked against the sha256 recorded in Week 4 before it is uploaded, and every upload is
checked against Hugging Face's copy afterwards. Nothing is trained or converted here.

**Before running:** Accelerator **None** · Internet **On** · Inputs: the **WEEK4** notebook output and the
**finetuning** notebook output · Add-ons → Secrets: **HF_TOKEN** (a *Write* token), attached to this notebook.

In [ ]:
# 1) Settings + the exact library versions this notebook was tested with.
%pip install -q "huggingface_hub==1.33.0" "hf_xet==1.7.0"
HF_USER    = "shubham-alyan24"
REPO_NAME  = "SmolLM3-3B-logtriage-GGUF"
PRIVATE    = True          # keep True. Make it public later on the Hugging Face website, after checking it.
GITHUB_URL = "https://github.com/shubham-alyan24/log-triage-slm"
INPUT_DIR  = "/kaggle/input"
OUT        = "/kaggle/working"
REPO_ID    = f"{HF_USER}/{REPO_NAME}"

import os, json, hashlib, time
import huggingface_hub
print("huggingface_hub", huggingface_hub.__version__, "| repo:", REPO_ID, "| private:", PRIVATE)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(16 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def find_one(name):
    hits = [os.path.join(d, name) for d, _, files in os.walk(INPUT_DIR) if name in files]
    if len(hits) != 1:
        raise SystemExit(f"Expected exactly 1 {name} under {INPUT_DIR}, found {len(hits)}: {hits}\n"
                         "Inputs needed: the WEEK4 notebook output and the finetuning notebook output.")
    return hits[0]

In [ ]:
# 2) Find every file and check it is byte-for-byte the file recorded in Week 4.
t = time.time()
INFO_PATH = find_one("week4_quant_info.json")
INFO = json.load(open(INFO_PATH))
W4_DIR = os.path.dirname(INFO_PATH)
Q8 = find_one("smollm3-logtriage-q8_0.gguf")
W3_DIR = os.path.dirname(Q8)

LEVELS = ["F16", "Q8_0", "Q5_K_M", "Q4_K_M", "Q3_K_M"]
GGUF = {}
for level in LEVELS:
    key = "Q8_0 (Week 3, reused)" if level == "Q8_0" else level
    rec = INFO["files"][key]
    path = Q8 if level == "Q8_0" else os.path.join(W4_DIR, rec["file"])
    got = sha256(path)
    ok = got == rec["sha256"] and os.path.getsize(path) == rec["bytes"]
    print(f"{level:7s} {rec['file']:34s} {rec['bytes']:>14,d} bytes  {'OK' if ok else 'MISMATCH'}")
    if not ok:
        raise SystemExit(f"{level}: file differs from week4_quant_info.json - paste this to Claude.")
    GGUF[level] = dict(rec, path=path, sha256=got)

ADAPTER_DIR = os.path.join(W3_DIR, "adapter")
ADAPTER_FILES = ["adapter_config.json", "adapter_model.safetensors"]
for name in ADAPTER_FILES:
    if not os.path.exists(os.path.join(ADAPTER_DIR, name)):
        raise SystemExit(f"Missing {name} in {ADAPTER_DIR}: {sorted(os.listdir(ADAPTER_DIR)) if os.path.isdir(ADAPTER_DIR) else 'no adapter folder'}")
ADAPTER_CFG = json.load(open(os.path.join(ADAPTER_DIR, "adapter_config.json")))
print("adapter base model:", ADAPTER_CFG.get("base_model_name_or_path"), "| r =", ADAPTER_CFG.get("r"),
      "| alpha =", ADAPTER_CFG.get("lora_alpha"))
if ADAPTER_CFG.get("r") != 16 or ADAPTER_CFG.get("lora_alpha") != 32:
    raise SystemExit("Adapter settings are not r=16 / alpha=32 - wrong adapter? Paste this to Claude.")
RUN_INFO = os.path.join(W3_DIR, "run_info.json")
WEIGHTS = INFO["bf16_to_f16"]["weights"]
print(f"All files verified in {time.time() - t:.0f} s. Weights per model: {WEIGHTS:,}")

In [ ]:
# 3) Write the model card (README.md). Results = frozen eval set, n=329, all scored on Ollama 0.34.2,
#    raw mode, temperature 0 (results/scores_eval_*.json in the GitHub repo).
RESULTS = [  # (model, level, action acc, action macro-F1, severity, service, id F1, check_network, escalate_oncall)
    ("Most-common answer", "-",      85.7, 23.1, 84.5, 23.1, None, "0/16",  "0/6"),
    ("Untuned SmolLM3-3B", "Q8_0",   83.3, 24.3, 66.6, 20.1, 29.0, "0/16",  "0/6"),
    ("Fine-tuned",         "F16",    97.0, 88.2, 95.7, 79.9, 91.9, "16/16", "6/6"),
    ("Fine-tuned",         "Q8_0",   97.0, 88.2, 95.7, 79.9, 92.2, "16/16", "6/6"),
    ("Fine-tuned",         "Q5_K_M", 97.3, 89.5, 95.7, 79.9, 92.0, "16/16", "6/6"),
    ("Fine-tuned",         "Q4_K_M", 96.0, 80.5, 95.7, 79.3, 91.3, "16/16", "3/6"),
    ("Fine-tuned",         "Q3_K_M", 93.9, 77.7, 93.6, 80.9, 89.2, "16/16", "4/6"),
]
def bpw(level):
    return GGUF[level]["bytes"] * 8 / WEIGHTS

files_rows = "\n".join(
    f"| `{GGUF[l]['file']}` | {l} | {GGUF[l]['bytes'] / 1e9:.2f} GB | {bpw(l):.2f} |" for l in LEVELS)
results_rows = "\n".join(
    f"| {m} | {lv} | {a:.1f} | **{f1:.1f}** | {s:.1f} | {sv:.1f} | {'-' if i is None else f'{i:.1f}'} | {cn} | **{eo}** |"
    for m, lv, a, f1, s, sv, i, cn, eo in RESULTS)
hashes = "\n".join(f"{GGUF[l]['sha256']}  {GGUF[l]['file']}" for l in LEVELS)

CARD = f"""---
license: apache-2.0
base_model: HuggingFaceTB/SmolLM3-3B
base_model_relation: finetune
pipeline_tag: text-generation
language:
- en
tags:
- gguf
- llama.cpp
- smollm3
- qlora
- log-analysis
- quantization
---

# SmolLM3-3B log triage (GGUF)

A QLoRA fine-tune of [HuggingFaceTB/SmolLM3-3B](https://huggingface.co/HuggingFaceTB/SmolLM3-3B) that reads **one
log line** and returns JSON:

```json
{{"service": "...", "severity": "INFO|WARNING|ERROR|FATAL", "identifiers": ["..."], "recommended_action": "no_action|investigate|escalate_oncall|check_network"}}
```

It was built for a university project asking: *how much task accuracy survives post-training quantisation of a
fine-tuned small language model?* Code, data splits and every prediction file: {GITHUB_URL}

## Files

| File | Level | Size | Bits per weight |
|---|---|---|---|
{files_rows}

`adapter/` holds the LoRA adapter (apply it to the base model with PEFT). The K-quant levels are mixes: some
tensors are kept at higher precision, so e.g. "Q4_K_M" averages about {bpw('Q4_K_M'):.1f} bits per weight.

## Results (frozen eval set, 329 log lines)

All rows scored the same way: Ollama 0.34.2, raw mode, temperature 0, same prompt, same CPU-only machine.
Action macro-F1 and the two rare-class columns are the headline metrics.

| Model | Level | Action acc | Action macro-F1 | Severity | Service | Identifier F1 | check_network found | escalate_oncall found |
|---|---|---|---|---|---|---|---|---|
{results_rows}

- **F16, Q8_0 and Q5_K_M perform the same.** Q5_K_M is 64% smaller than F16 with no measurable loss
  (its slightly higher macro-F1 is 1-2 lines - noise).
- **Q4_K_M** is the first level that loses anything: it finds 3 of 6 `escalate_oncall` lines.
  With only 6 such lines this is a warning sign, not proof.
- **Q3_K_M** declines across most metrics; all 10 extra action errors are on BGL lines.

## How to use

The model was trained on one exact prompt, built by `build_prompt()` in `src/config.py` of the GitHub repo
(SmolLM3 chat format, reasoning off). Send that prompt as a **raw completion** (do not let the runtime apply
its own chat template), with greedy decoding (temperature 0) and stop at `<|im_end|>`. No BOS token is added
(llama.cpp's default for this tokenizer), which matches training.

## How it was made

- **Data:** LogHub-2k, HDFS + BGL + OpenStack + Spark (8,000 lines). Split **by log template** (seed 42), so
  eval templates marked "unseen" never appear in training. Training set: 1,057 examples after removing
  duplicates and capping very common no_action patterns.
- **Labels:** `service` = LogHub component, `severity` = LogHub level, `identifiers` = alignment with the
  LogHub event template, `recommended_action` = labelled per (template, severity) group by gpt-oss-20b
  (so the action field is distilled from a larger model).
- **Training:** QLoRA (4-bit NF4), LoRA r=16, alpha=32, dropout 0.05 on all linear layers, 3 epochs,
  lr 2e-4 cosine, effective batch 16, loss on the answer only. The checkpoint was chosen by dev loss only
  (epoch 1). About 30 minutes on one Kaggle T4.
- **Quantisation:** llama.cpp commit `{INFO['llama_cpp_commit']}`. F16 and Q8_0 made by
  `convert_hf_to_gguf.py` from the merged bf16 model; Q5_K_M / Q4_K_M / Q3_K_M by `llama-quantize` from the
  F16 file, without an importance matrix. A Q8_0 made the llama-quantize way is byte-identical to the one here.

## Limitations

- Small eval set (329 lines). `escalate_oncall` has only 6 eval lines and 11 distinct training lines, so
  results on it are anecdotal. All `check_network` training lines come from HDFS.
- The eval labels were reviewed by an AI model, not by hand.
- `service` is not written in the log text, so it is memorised: 59.7% on unseen templates is the ceiling of
  this setup.
- Only tested on these four LogHub systems. A research artefact, not a production triage system.

## sha256

```
{hashes}
```
"""
open(f"{OUT}/README.md", "w", encoding="utf-8", newline="\n").write(CARD)
print(CARD[:1500], "\n...\n(model card saved to", f"{OUT}/README.md)")

In [ ]:
# 4) Log in with the token from Kaggle Secrets (never paste the token into this notebook).
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, CommitOperationAdd
try:
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(f"No HF_TOKEN secret attached ({type(e).__name__}). Add-ons -> Secrets -> HF_TOKEN, tick it for this notebook.")
api = HfApi(token=TOKEN)
me = api.whoami()
role = me.get("auth", {}).get("accessToken", {}).get("role")
print("Logged in as:", me.get("name"), "| token type:", role)
if me.get("name") != HF_USER:
    raise SystemExit(f"Token belongs to {me.get('name')!r}, expected {HF_USER!r}.")
if role == "read":
    raise SystemExit("This is a READ token. Create a WRITE token on huggingface.co -> Settings -> Access Tokens.")

In [ ]:
# 5) Create the PRIVATE repo and upload everything in ONE commit (about 15 GB; roughly 5-20 min).
t = time.time()
api.create_repo(REPO_ID, private=PRIVATE, exist_ok=True)
UPLOADS = {"README.md": f"{OUT}/README.md",
           **{GGUF[l]["file"]: GGUF[l]["path"] for l in LEVELS},
           **{f"adapter/{n}": os.path.join(ADAPTER_DIR, n) for n in ADAPTER_FILES},
           "results/week4_quant_info.json": INFO_PATH,
           "results/week3_run_info.json": RUN_INFO}
for repo_path, local in UPLOADS.items():
    print(f"  {repo_path:40s} {os.path.getsize(local) / 1e9:8.3f} GB")
info = api.create_commit(REPO_ID, operations=[CommitOperationAdd(p, l) for p, l in UPLOADS.items()],
                         commit_message="Fine-tuned SmolLM3-3B log triage: 5 GGUF levels, LoRA adapter, model card")
print(f"Uploaded in {(time.time() - t) / 60:.1f} min:", info.commit_url)

In [ ]:
# 6) Check Hugging Face's copy: every file present, same size, and same sha256 where Hugging Face reports it.
remote = {f.path: f for f in api.list_repo_tree(REPO_ID, recursive=True, expand=True) if hasattr(f, "size")}
problems = []
for repo_path, local in UPLOADS.items():
    r = remote.get(repo_path)
    if r is None:
        problems.append(f"{repo_path}: missing on Hugging Face")
        continue
    size_ok = r.size == os.path.getsize(local)
    lfs_sha = getattr(getattr(r, "lfs", None), "sha256", None)
    sha_ok = None if lfs_sha is None else lfs_sha == sha256(local)
    print(f"  {repo_path:40s} size {'OK' if size_ok else 'DIFFERENT'}  sha256 "
          f"{'not reported' if sha_ok is None else ('OK' if sha_ok else 'DIFFERENT')}")
    if not size_ok or sha_ok is False:
        problems.append(repo_path)
if problems:
    raise SystemExit(f"Upload check failed for {problems} - paste this to Claude.")
print(f"\nDONE. All {len(UPLOADS)} files are on Hugging Face (private): https://huggingface.co/{REPO_ID}")
print("Check the page, then make it public: repo Settings -> Change repository visibility.")